In [1]:
import pandas as pd
import numpy as np
import csv

In [5]:
countries = ["Germany", "Spain", "Romania"]
year = 2025

for country in countries:
    filename = f"{country}_Production_Type_{year}_Generation.csv"
    consumption_filename = f"{country}_Production_Type_{year}_Consumption.csv"

    # Read raw data
    df = pd.read_csv(filename)

    # Mapping from Production Type to target column names
    production_mapping = {
        "Biomass": "Biomass - Actual Aggregated [MW]",
        "Energy storage": "Energy storage - Actual Aggregated [MW]",
        "Fossil Brown coal/Lignite": "Fossil Brown coal/Lignite - Actual Aggregated [MW]",
        "Fossil Coal-derived gas": "Fossil Coal-derived gas - Actual Aggregated [MW]",
        "Fossil Gas": "Fossil Gas - Actual Aggregated [MW]",
        "Fossil Hard coal": "Fossil Hard coal - Actual Aggregated [MW]",
        "Fossil Oil": "Fossil Oil - Actual Aggregated [MW]",
        "Fossil Oil shale": "Fossil Oil shale - Actual Aggregated [MW]",
        "Fossil Peat": "Fossil Peat - Actual Aggregated [MW]",
        "Geothermal": "Geothermal - Actual Aggregated [MW]",
        "Hydro Pumped Storage": "Hydro Pumped Storage - Actual Aggregated [MW]",
        "Hydro Pumped Storage Consumption": "Hydro Pumped Storage - Actual Consumption [MW]",
        "Hydro Run-of-river and pondage": "Hydro Run-of-river and poundage - Actual Aggregated [MW]",
        "Hydro Water Reservoir": "Hydro Water Reservoir - Actual Aggregated [MW]",
        "Marine": "Marine - Actual Aggregated [MW]",
        "Nuclear": "Nuclear - Actual Aggregated [MW]",
        "Other": "Other - Actual Aggregated [MW]",
        "Other renewable": "Other renewable - Actual Aggregated [MW]",
        "Solar": "Solar - Actual Aggregated [MW]",
        "Waste": "Waste - Actual Aggregated [MW]",
        "Wind Offshore": "Wind Offshore - Actual Aggregated [MW]",
        "Wind Onshore": "Wind Onshore - Actual Aggregated [MW]",
    }

    # Rename production types
    df["Production Type"] = df["Production Type"].replace(production_mapping)

    # Pivot
    pivot = df.pivot_table(
        index=["Area", "MTU (UTC)"],
        columns="Production Type",
        values="Generation (MW)",
        aggfunc="first"
    )

    pivot = pivot.reset_index()


    # Rename MTU column
    pivot = pivot.rename(columns={"MTU (UTC)": "MTU"})

    def format_mtu(mtu):
        start, end = mtu.split(" - ")
        start = pd.to_datetime(start, format="%d/%m/%Y %H:%M:%S")
        end = pd.to_datetime(end, format="%d/%m/%Y %H:%M:%S")

        return (
            f"{start.strftime('%d.%m.%Y %H:%M')} - "
            f"{end.strftime('%d.%m.%Y %H:%M')} (UTC)"
        )

    pivot["MTU"] = pivot["MTU"].apply(format_mtu)
    # Desired output order
    target_columns = [
        "Area",
        "MTU",
        "Biomass - Actual Aggregated [MW]",
        "Energy storage - Actual Aggregated [MW]",
        "Fossil Brown coal/Lignite - Actual Aggregated [MW]",
        "Fossil Coal-derived gas - Actual Aggregated [MW]",
        "Fossil Gas - Actual Aggregated [MW]",
        "Fossil Hard coal - Actual Aggregated [MW]",
        "Fossil Oil - Actual Aggregated [MW]",
        "Fossil Oil shale - Actual Aggregated [MW]",
        "Fossil Peat - Actual Aggregated [MW]",
        "Geothermal - Actual Aggregated [MW]",
        "Hydro Pumped Storage - Actual Aggregated [MW]",
        "Hydro Pumped Storage - Actual Consumption [MW]",
        "Hydro Run-of-river and poundage - Actual Aggregated [MW]",
        "Hydro Water Reservoir - Actual Aggregated [MW]",
        "Marine - Actual Aggregated [MW]",
        "Nuclear - Actual Aggregated [MW]",
        "Other - Actual Aggregated [MW]",
        "Other renewable - Actual Aggregated [MW]",
        "Solar - Actual Aggregated [MW]",
        "Waste - Actual Aggregated [MW]",
        "Wind Offshore - Actual Aggregated [MW]",
        "Wind Onshore - Actual Aggregated [MW]",
    ]

    # Add missing columns as n/e
    for col in target_columns:
        if col not in pivot.columns:
            pivot[col] = "n/e"

    # Reorder columns
    pivot = pivot[target_columns]

    # Keep values as strings, rounded to nearest integer
    for col in target_columns[2:]:
        pivot[col] = pivot[col].apply(
            lambda x: str(round(float(x))) if pd.notna(x) and x != "n/e" else "n/e"
        )

    cons = pd.read_csv(consumption_filename)
    cons["Production Type"] = cons["Production Type"].replace({"Hydro Pumped Storage": "Hydro Pumped Storage - Actual Consumption [MW]"})
    cons_pivot = cons.pivot_table(
        index=["Area", "MTU (UTC)"],
        columns="Production Type",
        values="Consumption (MW)",
        aggfunc="first"
    )
    cons_pivot = cons_pivot.reset_index()
    cons_pivot = cons_pivot.rename(columns={"MTU (UTC)": "MTU"})
    # Format MTU the same way if needed
    cons_pivot["MTU"] = cons_pivot["MTU"].apply(format_mtu)

    # Keep as rounded strings
    cons_pivot["Hydro Pumped Storage - Actual Consumption [MW]"] = (cons_pivot["Hydro Pumped Storage - Actual Consumption [MW]"]
                                                            .apply(lambda x: str(round(float(x))) if pd.notna(x) and x != "n/e" else "n/e")
                                                            )

    pivot["Hydro Pumped Storage - Actual Consumption [MW]"] = (
        cons_pivot["Hydro Pumped Storage - Actual Consumption [MW]"]
        .fillna("n/e")
    )
    import pandas as pd

    def fix_ne_vs_values(df):
        for col in df.columns:
            if col in ["Area", "MTU"]:
                continue

            series = df[col]

            has_ne = (series == "n/e").any()
            has_numbers = pd.to_numeric(series, errors="coerce").notna().any()

            if has_ne and has_numbers:
                df[col] = series.replace("n/e", "0")

        return df
    pivot = fix_ne_vs_values(pivot)
    # pivot.to_csv(f"{country}_Generation_per_Production_Type_{year}.csv", index=False)
    pivot.to_csv(
    f"{country}_Generation_per_Production_Type_{year}.csv",
    index=False,
    quoting=csv.QUOTE_ALL)

/tmp/ipykernel_261189/2705636446.py:9: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filename)
/tmp/ipykernel_261189/2705636446.py:9: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filename)
